# Set Up


## Libraries

In [1]:
# Standard Python libraries
import re
import unicodedata
import os

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

## Paths

In [2]:
# S3 project paths
S3_BUCKET = "s3a://dsan6000-jg2292/ancient-battles"

RAW_FILE = f"{S3_BUCKET}/Raw/articles.jsonl"
CLEANED_DIR = f"{S3_BUCKET}/processed/cleaned_articles"

# Primary corpus settings
MAX_CRAWL_DEPTH = 1
EXPECTED_ARTICLES = 9015

print(f"Raw input: {RAW_FILE}")
print(f"Cleaned output: {CLEANED_DIR}")
print(f"Maximum crawl depth: {MAX_CRAWL_DEPTH}")

Raw input: s3a://dsan6000-jg2292/ancient-battles/Raw/articles.jsonl
Cleaned output: s3a://dsan6000-jg2292/ancient-battles/processed/cleaned_articles
Maximum crawl depth: 1


# Initialize Spark

In [3]:
# Spark configuration
SPARK_APP_NAME = "Ancient Battles - Text Cleaning"

SPARK_MASTER = "spark://ip-172-31-46-25.ec2.internal:7077"

# Initialize Spark
spark = (
    SparkSession.builder
    .appName(SPARK_APP_NAME)
    .master(SPARK_MASTER)
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.5.0"
    )
    .getOrCreate()
)

spark.conf.set(
    "spark.sql.execution.pythonUDF.arrow.enabled",
    "false"
)

# Display Spark configuration
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)

:: loading settings :: url = jar:file:/home/ubuntu/dsan6000/DSAN600-AncientBattles/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ubuntu/.ivy2.5.2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-532346da-1568-40c4-8216-852a25a2ce3a;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 443ms :: artifacts dl 12ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35

Spark version: 4.3.0
Spark master: spark://ip-172-31-46-25.ec2.internal:7077
Application ID: app-20261115211939-0002


## Verify distributed execution

In [4]:
# Create a small distributed dataset
numbers = spark.sparkContext.parallelize(range(1, 101), 6)

# Perform a distributed calculation
total = numbers.sum()

print("Sum:", total)
print("Number of partitions:", numbers.getNumPartitions())
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Sum: 5050
Number of partitions: 6
Default parallelism: 6


# Text Preprocessing

## Define the input schema

In [5]:
# Define the schema for the fields needed in text preprocessing
article_schema = T.StructType([
    T.StructField("page_id", T.LongType(), True),
    T.StructField("title", T.StringType(), True),
    T.StructField("text", T.StringType(), True),
    T.StructField("crawl_depth", T.IntegerType(), True)
])

## Read the raw JSONL file

In [6]:
# Read the raw Wikipedia articles from S3
raw_articles_df = (
    spark.read
    .schema(article_schema)
    .json(RAW_FILE)
)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Small Check 
The above was having schema issues previously so this is to double check that

In [7]:
print("Schema:")
raw_articles_df.printSchema()

Schema:
root
 |-- page_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- crawl_depth: integer (nullable = true)



## Verify Corpus Size

In [8]:
# Filter to seed articles and first-level linked articles
articles_df = raw_articles_df.filter(
    F.col("crawl_depth") <= MAX_CRAWL_DEPTH
)

# Count primary corpus articles
print("Primary corpus articles:", articles_df.count())

Primary corpus articles: 9015


# Text Cleaning

## Set Up and Check 

In [9]:
# Required columns
required_columns = {
    "page_id",
    "title",
    "text",
    "crawl_depth"
}

missing_columns = required_columns - set(articles_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Check for missing article text
missing_text = (
    articles_df
    .filter(F.col("text").isNull())
    .count()
)

print("Articles in primary corpus:", articles_df.count())
print("Articles with missing text:", missing_text)

Articles in primary corpus: 9015
Articles with missing text: 0


## Cleaning 

### Cleaning Function

In [10]:
def clean_article_text(text):
    """
    Apply conservative formatting cleanup to one Wikipedia article.
    Remove references and bibliography sections and everything after them.
    """

    if text is None:
        return ""

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Standardize whitespace and apostrophes
    text = text.replace("\u00a0", " ")
    text = text.replace("’", "'")

    # Identify headings that indicate the beginning of references
    reference_headings = {
        "references",
        "bibliography",
        "citations",
        "notes and references",
        "notes",
        "sources"
    }

    # Split article into lines
    lines = text.splitlines()

    cleaned_lines = []

    for line in lines:

        # Normalize heading text for comparison
        heading = line.strip().strip("=").strip()
        heading = heading.lower()

        # Stop processing when a reference-related heading is found
        if heading in reference_headings:
            break

        cleaned_lines.append(line)

    # Reassemble article text
    text = "\n".join(cleaned_lines)

    # Remove numeric citation markers
    text = re.sub(
        r"\[\s*\d+(?:\s*[,;–-]\s*\d+)*\s*\]",
        " ",
        text
    )

    # Remove common editorial markers
    text = re.sub(
        r"\[(?:citation needed|clarification needed|verification needed)\]",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove URLs
    text = re.sub(r"https?://\S+", " ", text)

    # Remove residual HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize whitespace while preserving paragraph breaks
    lines = []

    for line in text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()

        if line:
            lines.append(line)

    text = "\n\n".join(lines)

    return text.strip()

### Register the cleaning function as a Spark UDF

In [11]:
clean_article_udf = F.udf(
    clean_article_text,
    T.StringType()
)

### Test Function

In [12]:
# Select one article
example_row = (
    articles_df
    .select("title", "text")
    .first()
)

example_title = example_row["title"]
example_text = example_row["text"]

# Apply cleaning
example_cleaned_text = clean_article_text(example_text)

print("Article:", example_title)

print("\n--- ORIGINAL TEXT ---\n")
print(example_text[:2000])

print("\n--- CLEANED TEXT ---\n")
print(example_cleaned_text[:2000])

print("\nOriginal characters:", len(example_text))
print("Cleaned characters:", len(example_cleaned_text))
print(
    "Characters removed:",
    len(example_text) - len(example_cleaned_text)
)

Article: Battle of Arbalo

--- ORIGINAL TEXT ---

The Battle of Arbalo was fought between the Romans and Germanic peoples in 11 BC in a narrow pass called Arbalo somewhere west of the Weser and probably near the Lippe.
It was part of the Drusus' campaigns (12–8 BC), which started in the unstable north of the Roman Empire near Gaul. With the Germanic from the east of Gaul constantly attacking Rome, Augustus would send Drusus.
As part of operations by Augustus to secure the borders of the Roman Empire, Drusus, military commander and stepson of Augustus, was given the order to pacify the region on either side of the River Rhine. In spring 11 BC. he crossed the Rhine with his army and defeated the Usipetes. He built a bridge over the Lippe and marched through the territory of the Sugambri and the Cherusci to the Weser.

During its return march to the Rhine the army was lured into an ambush at a place called Arbalo. It was attacked in a narrow pass by Cherusci.
Using the element of surprise

## Apply cleaning to the entire corpus

In [13]:
# Apply the cleaning function to every article
cleaned_articles_df = (
    articles_df
    .withColumn(
        "cleaned_text",
        clean_article_udf(F.col("text"))
    )
)

print("Cleaning transformation defined.")

Cleaning transformation defined.


### Calculate text-length statistics

In [14]:
# Calculate original and cleaned text lengths
cleaned_articles_df = (
    cleaned_articles_df
    .withColumn(
        "original_characters",
        F.length(F.col("text"))
    )
    .withColumn(
        "cleaned_characters",
        F.length(F.col("cleaned_text"))
    )
    .withColumn(
        "characters_removed",
        F.col("original_characters") -
        F.col("cleaned_characters")
    )
)

print("Character statistics added.")

Character statistics added.


### Save cleaned articles to S3

In [15]:
# Select columns to retain in the processed dataset
processed_articles_df = cleaned_articles_df.select(
    "page_id",
    "title",
    "crawl_depth",
    "cleaned_text",
    "original_characters",
    "cleaned_characters",
    "characters_removed"
)

# Save processed articles to S3 as Parquet
(
    processed_articles_df
    .write
    .mode("overwrite")
    .parquet(CLEANED_DIR)
)

print("Processed articles saved to:", CLEANED_DIR)

Processed articles saved to: s3a://dsan6000-jg2292/ancient-battles/processed/cleaned_articles


In [18]:
print(
    "Arrow optimization for Python UDFs:",
    spark.conf.get("spark.sql.execution.pythonUDF.arrow.enabled")
)

Arrow optimization for Python UDFs: true


### Verify saved articles

In [16]:
# Read the saved dataset from S3
saved_articles_df = spark.read.parquet(CLEANED_DIR)

# Verify the saved article count
article_count = saved_articles_df.count()

print(f"Articles saved: {article_count:,}")
print(f"Expected articles: {EXPECTED_ARTICLES:,}")

# Verify the saved dataset schema
saved_articles_df.printSchema()

if article_count != EXPECTED_ARTICLES:
    raise ValueError("Saved article count does not match expected count.")

Articles saved: 9,015
Expected articles: 9,015
root
 |-- page_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- crawl_depth: integer (nullable = true)
 |-- cleaned_text: string (nullable = true)
 |-- original_characters: integer (nullable = true)
 |-- cleaned_characters: integer (nullable = true)
 |-- characters_removed: integer (nullable = true)



### Inspect the cleaning results

In [17]:
# Identify articles with the most characters removed
cleaning_summary_df = (
    saved_articles_df
    .select(
        "title",
        "page_id",
        "original_characters",
        "cleaned_characters",
        "characters_removed"
    )
    .orderBy(
        F.col("characters_removed").desc()
    )
)

cleaning_summary_df.show(10, truncate=False)

+---------------------+--------+-------------------+------------------+------------------+
|title                |page_id |original_characters|cleaned_characters|characters_removed|
+---------------------+--------+-------------------+------------------+------------------+
|Late Roman army      |14531824|128725             |3830              |124895            |
|Ancient Carthage     |20663625|121431             |4972              |116459            |
|Hellenistic period   |455379  |101550             |4712              |96838             |
|Constantine the Great|7236    |79440              |4637              |74803             |
|Trajan               |30570   |73646              |2066              |71580             |
|Antony and Cleopatra |65169   |77538              |7694              |69844             |
|Indo-Greek Kingdom   |779319  |86756              |22589             |64167             |
|Dacian language      |384709  |64648              |679               |63969             |

### Calculate overall cleaning impact

In [18]:
# Calculate overall cleaning statistics
cleaning_totals = (
    saved_articles_df
    .agg(
        F.sum("original_characters").alias("total_original"),
        F.sum("cleaned_characters").alias("total_cleaned"),
        F.sum("characters_removed").alias("total_removed")
    )
    .first()
)

total_original = cleaning_totals["total_original"] or 0
total_cleaned = cleaning_totals["total_cleaned"] or 0
total_removed = cleaning_totals["total_removed"] or 0

percentage_removed = (
    (total_removed / total_original) * 100
    if total_original > 0
    else 0
)

print(f"Total original characters: {total_original:,}")
print(f"Total cleaned characters: {total_cleaned:,}")
print(f"Total characters removed: {total_removed:,}")
print(f"Percentage removed: {percentage_removed:.2f}%")

Total original characters: 126,251,439
Total cleaned characters: 117,789,767
Total characters removed: 8,461,672
Percentage removed: 6.70%
